# Module 06: Code Quality

AI as your code reviewer.

## Learning Objectives

1. Use formatters (Black, Ruff) for consistent style
2. Apply linters (Flake8, Pylint) for quality checks
3. Set up pre-commit hooks
4. Use AI for code review and refactoring

## Why Code Quality Tools?

AI generates code quickly, but not always cleanly:
- Inconsistent formatting
- Unused imports
- Missing docstrings
- Style violations

**Automation** catches these issues before commit.

## Black: The Formatter

Black formats code automatically - no decisions needed.

```bash
# Install
pip install black

# Format file
black myfile.py

# Format directory
black src/

# Check without changing
black --check src/
```

Configure in pyproject.toml:
```toml
[tool.black]
line-length = 88
target-version = ['py310']
```

## Exercise 1: Format Some Code (5 min)

Create a file called `messy.py` with this intentionally ugly code:

```python
x=1;y=2;z=x+y
def   f(  a,b,c ):
    return(a+b+c)
data = {'key1':1,'key2':2,'key3':3,'key4':4,'key5':5,'key6':6,'key7':7}
```

**Step 1:** Preview what Black would change (without modifying the file):

```bash
black --diff messy.py
```

Read the diff carefully. What changes does Black want to make?

**Step 2:** Apply the formatting:

```bash
black messy.py
```

**Step 3:** Open `messy.py` and inspect the result.

**Questions to answer:**
- Did Black split the dictionary across multiple lines? Why?
- Did Black remove the semicolons?
- Was anything surprising about the formatting choices?
- What happened to the extra spaces in the function definition?

## Ruff: The Fast Linter

Ruff is a fast Python linter (replaces Flake8, isort, and more):

```bash
pip install ruff

# Check
ruff check src/

# Fix automatically
ruff check --fix src/
```

Configure in pyproject.toml:
```toml
[tool.ruff]
line-length = 88
select = ["E", "F", "I", "D"]  # Errors, pyflakes, isort, docstrings
ignore = ["D100", "D104"]  # Ignore specific rules
```

## Exercise 2: Lint and Fix (10 min)

Create a file called `lint_me.py` with this code that has several quality issues:

```python
import os
import sys
import json

def processData(input):
    x = json.loads(input)
    return x
```

**Step 1:** Run Ruff on it:

```bash
ruff check lint_me.py
```

**Step 2:** Answer these questions:
- How many issues were found?
- What do the error codes (e.g., F401, E741) mean? (Hint: `ruff rule F401`)
- Which imports are flagged as unused?
- Is `input` flagged? Why? (Hint: it shadows a Python builtin.)

**Step 3:** Let Ruff auto-fix what it can:

```bash
ruff check --fix lint_me.py
```

**Step 4:** Check the file again. Which issues did Ruff fix automatically? Which require you to fix manually? Fix the remaining issues by hand (rename `input` to something like `data_str`, rename `processData` to `process_data` following PEP 8).

## Pre-commit Hooks

Run checks automatically before each commit:

```bash
pip install pre-commit
```

Create `.pre-commit-config.yaml`:
```yaml
repos:
  - repo: https://github.com/psf/black
    rev: 23.12.0
    hooks:
      - id: black

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.1.8
    hooks:
      - id: ruff
        args: [--fix]

  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.5.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
```

Install hooks:
```bash
pre-commit install
```

Now hooks run on every commit!

## Exercise 3: Set Up Pre-commit (10 min)

Experience what automated quality control feels like in a real workflow.

**Step 1:** Create a temporary git repository to experiment in:

```bash
mkdir /tmp/precommit-demo && cd /tmp/precommit-demo
git init
```

**Step 2:** Install pre-commit and create the config file:

```bash
pip install pre-commit
```

Create `.pre-commit-config.yaml` with the content shown above (Black, Ruff, and the basic hooks).

**Step 3:** Install the hooks:

```bash
pre-commit install
```

**Step 4:** Create a deliberately messy Python file:

```bash
cat > bad_code.py << 'EOF'
import os
import sys
x=1;y=2
def f( a,b ):
    return a+b
EOF
```

**Step 5:** Try to commit it:

```bash
git add bad_code.py .pre-commit-config.yaml
git commit -m "add code"
```

Watch the hooks run and catch issues. Some hooks (like Black) will auto-fix the file.

**Step 6:** After the hooks modify the file, `git add` the fixed version and commit again. This time it should succeed.

**Key takeaway:** Pre-commit hooks enforce quality automatically — you cannot commit code that violates the rules.

## AI Code Review

Use AI as an additional reviewer:

```
> Review this code for:
  - Security issues
  - Performance problems
  - Python best practices
  - Potential bugs

[paste code]
```

AI might find:
- SQL injection vulnerabilities
- Inefficient loops
- Missing error handling
- Non-idiomatic patterns

## AI Refactoring

```
> Refactor this function to:
  - Use list comprehension instead of loops
  - Add type hints
  - Add a docstring with examples
  - Handle edge cases
```

Before:
```python
def filter_papers(papers, min_citations):
    result = []
    for p in papers:
        if p['citations'] >= min_citations:
            result.append(p)
    return result
```

After AI refactoring:
```python
def filter_papers(
    papers: list[dict], 
    min_citations: int = 0
) -> list[dict]:
    """Filter papers by minimum citation count.
    
    Examples:
        >>> filter_papers([{'citations': 10}], 5)
        [{'citations': 10}]
    """
    if not papers:
        return []
    return [p for p in papers if p.get('citations', 0) >= min_citations]
```

## Exercise 4: AI Code Review (10 min)

Use AI as a code reviewer on a function with multiple quality issues.

**Step 1:** Take this function:

```python
def get_stuff(d, k):
    r = []
    for i in range(len(d)):
        if d[i][k] > 0:
            r.append(d[i])
    return r
```

**Step 2:** Ask your AI agent:

> "Review this code for quality issues and suggest improvements:
> ```python
> def get_stuff(d, k):
>     r = []
>     for i in range(len(d)):
>         if d[i][k] > 0:
>             r.append(d[i])
>     return r
> ```"

**Step 3:** Compare the AI's suggestions to this checklist of issues:

- [ ] **Naming:** `get_stuff`, `d`, `k`, `r` are all unclear names
- [ ] **Iteration:** `for i in range(len(d))` should be `for item in d`
- [ ] **Pattern:** The loop-append pattern should be a list comprehension
- [ ] **Type hints:** No type annotations
- [ ] **Docstring:** No documentation
- [ ] **Edge cases:** What if `d` is empty? What if a dict is missing key `k`?

How many did AI catch? Did AI suggest anything you didn't think of?

**Step 4:** Apply the AI's suggestions (or write your own improved version). A good refactored version might look like:

```python
def filter_positive(records: list[dict], key: str) -> list[dict]:
    """Filter records where the given key has a positive value."""
    return [record for record in records if record.get(key, 0) > 0]
```

Which version would you rather maintain six months from now?

## Exercise: Set Up Quality Pipeline

1. Add Black and Ruff to your package
2. Configure in pyproject.toml
3. Set up pre-commit hooks
4. Run on existing code, fix issues
5. Have AI review your code

Use AI to generate the configuration files.

## Summary

| Tool | Purpose |
|------|--------|
| Black | Auto-formatting |
| Ruff | Fast linting |
| pre-commit | Automated checks |
| AI review | Deep analysis |

## Next Steps

In the next module, we'll learn Python classes and object-oriented design.